In [ ]:
!pip install -q langgraph langgraph-checkpoint-sqlite google-genai

In [ ]:
import re
import httpx
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

_ = load_dotenv()

In [ ]:
import json
import time
import sqlite3
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver

In [ ]:
api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass

if not api_key:
    raise ValueError("No API key found. Set GEMINI_API_KEY as an environment variable or a Colab secret.")

client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-3.6-flash"

print("Client initialized.")

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DB_DIR = "/content/drive/MyDrive/dd_agent"
except Exception:
    DB_DIR = "."
os.makedirs(DB_DIR, exist_ok=True)
DB_PATH = os.path.join(DB_DIR, "dd_checkpoints.db")
print("Checkpoint DB:", DB_PATH)

In [ ]:
import os

REPO = "10k-financial-analyst-agent"
if not os.path.exists(REPO):
    !git clone https://github.com/mzs200606/{REPO}.git
else:
    !git -C {REPO} pull
%cd {REPO}

from ratios import FIELDS, REQUIRED, OPTIONAL, with_derived, compute_ratio, compute_all_ratios
from thresholds import THRESHOLDS, SECTORS, classify
from validation import validate_figures
from flags import trend_flags

In [ ]:
import agent

graph = agent.build_graph(client, MODEL_ID, DB_PATH)
print("Graph compiled.")

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Could not render the diagram image (needs internet access to mermaid.ink):", e)
    print(graph.get_graph().draw_mermaid())

In [ ]:
ICON = {"GREEN": "🟢", "RED": "🔴", "NEUTRAL": "⚪", "N/A": "➖"}


def run(inputs, config):
    """Stream the graph and print a line when each node finishes."""
    for event in graph.stream(inputs, config):
        for node in event:
            if not node.startswith("__"):
                print(f"\n✔ finished node: {node}")


def fmt(v):
    return "—" if v is None else f"{v:,.2f}".rstrip("0").rstrip(".")


def show_review(thread):
    s = graph.get_state(thread).values
    print(f"\n=== REVIEW: {s['company']} | {s['fy_current']} vs {s['fy_prior']} | units: {s['units']} {s['currency']} | sector rules: {s['sector']} ===")
    print(f"{'figure':<30}{'current':>16}{'pg':>8}{'prior':>16}{'pg':>8}")
    for k in FIELDS:
        print(f"{k:<30}{fmt(s['figures']['current'][k]):>16}{str(s['pages']['current'][k] or '—'):>8}"
              f"{fmt(s['figures']['prior'][k]):>16}{str(s['pages']['prior'][k] or '—'):>8}")
    cur_d = with_derived(s["figures"]["current"])
    pri_d = with_derived(s["figures"]["prior"])
    for k in ("interest_expense", "short_term_debt", "total_debt"):
        print(f"{k + ' (derived)':<30}{fmt(cur_d[k]):>16}{'':>8}{fmt(pri_d[k]):>16}")
    print("\nValidation:", "none, all checks passed" if not s["issues"] else "")
    for i in s["issues"]:
        print("  -", i)


def fix_figure(thread, period, key, value):
    """Correct one figure and re-run validation (the logic lives in agent.py)."""
    agent.fix_figure(graph, thread, period, key, value)


def review_and_resume(thread):
    """Interactive loop like the course's 'proceed?' loop: y = approve and continue, e = edit a figure, q = quit."""
    while True:
        snap = graph.get_state(thread)
        show_review(thread)
        blockers = [i for i in snap.values["issues"] if i.startswith("BLOCKER")]
        if blockers:
            choice = input("\nBlocked. [e] edit a figure, [q] quit: ").strip().lower()
        elif snap.next == ("compute_ratios",):
            choice = input("\nApprove these figures? [y] yes and continue, [e] edit a figure, [q] quit: ").strip().lower()
        else:
            print("Nothing to review: the graph is not paused at the review step.")
            return
        if choice == "y" and not blockers:
            run(None, thread)
            return
        if choice == "e":
            period = input("  period (current/prior): ").strip().lower()
            key = input(f"  figure name ({', '.join(FIELDS)}): ").strip()
            raw = input("  new value (number, or 'null'): ").strip().replace(",", "")
            try:
                fix_figure(thread, period, key, None if raw.lower() == "null" else float(raw))
            except (AssertionError, ValueError) as e:
                print("  Could not apply that edit:", e)
        elif choice == "q":
            print("Stopped. The thread is saved; you can come back to it later.")
            return


def show_results(thread, show_memo=False):
    s = graph.get_state(thread).values
    if "verdicts" not in s:
        print("No results yet for this thread.")
        return
    print(f"\n=== RESULTS: {s['company']} | {s['fy_current']} | sector rules: {s['sector']} ===")
    print(f"{'ratio':<20}{'current':>10}{'prior':>10}   verdict   rule")
    for name, v in s["verdicts"].items():
        extra = f"  ({v['note']})" if v["note"] else ""
        print(f"{name:<20}{fmt(v['value']):>10}{fmt(v['prior_value']):>10}   {ICON[v['verdict']]} {v['verdict']:<8} {v['rule']}{extra}")
    print("\nRed flags:", "none" if not s["flags"] else "")
    for f in s["flags"]:
        print("  🚩", f)
    if show_memo:
        print("\n" + s["memo"])

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
except Exception:
    pdf_path = "your_10k.pdf"
print("Using:", pdf_path)

In [ ]:
SECTOR = "general"                      # general | retail | tech | utilities_energy
THREAD_ID = "WMT-FY2025"         # change for every new filing
assert SECTOR in SECTORS, f"SECTOR must be one of {SECTORS}"

thread = {"configurable": {"thread_id": THREAD_ID}}
run({"pdf_path": pdf_path, "sector": SECTOR}, thread)
print("\nPaused at:", graph.get_state(thread).next)

In [ ]:
review_and_resume(thread)

In [ ]:
show_results(thread, show_memo=False)

## 6. OPTIONAL: time travel
Every run saved a checkpoint right before `compute_ratios`. These cells branch from that checkpoint, so you do **not** re-upload or re-extract.
The example re-scores the *same figures* under a different sector's thresholds.

In [ ]:
def review_points(thread):
    """Checkpoints where the graph was paused before compute_ratios, newest first."""
    pts = [s for s in graph.get_state_history(thread) if s.next == ("compute_ratios",)]
    for i, s in enumerate(pts):
        print(f"[{i}] step {s.metadata.get('step')} | sector={s.values.get('sector')} | issues={len(s.values.get('issues', []))}")
    return pts


def branch_from_review(thread, updates, which=0):
    """Fork from a review checkpoint with changed state (e.g. {'sector': 'retail'}), then run the rest of the graph."""
    pts = [s for s in graph.get_state_history(thread) if s.next == ("compute_ratios",)]
    base = pts[which]
    fork_config = graph.update_state(base.config, updates, as_node="validate")
    run(None, fork_config)
    return graph.get_state(thread).config


pts = review_points(thread)

## 7. OPTIONAL: persistence

In [ ]:
thread = {"configurable": {"thread_id": THREAD_ID}}
snap = graph.get_state(thread)
print("Company:", snap.values.get("company"), "| next step:", snap.next)
show_results(thread, show_memo=True)